# Workshop Magentic: Coordinación Clínica Iterativa (Hospital)

## 🎯 ¿Qué es el Patrón Magentic?

El patrón **Magentic** es una estrategia de orquestación donde un **gestor (manager)** coordina especialistas y puede **iterar en rondas**: planifica, delega a un especialista, evalúa el progreso y decide si continuar o finalizar.

A diferencia de Sequential (pasos fijos) o Concurrent (paralelo sin diálogo), Magentic es útil cuando necesitas **refinar** una decisión clínica hasta que sea segura y completa.


## 🏥 Escenario del ejercicio (muy concreto)

Estás en Urgencias y necesitas coordinar diagnóstico y tratamiento inicial ante un caso potencialmente grave:
> “Paciente de 54 años con fiebre alta, confusión y TA 85/55. Sospecha de sepsis.”

Lo que buscamos demostrar con **Magentic** es esto:
1. `CoordinadorClinico` define un plan corto (qué hay que resolver y en qué orden).
2. `Diagnostico` aporta hipótesis y pruebas/acciones diagnósticas iniciales.
3. `Tratamiento` propone medidas terapéuticas iniciales seguras (incluida vigilancia/contraindicaciones).
4. El manager integra y entrega una respuesta final accionable.

### ✅ Qué deberías ver en la salida
- Una secuencia de eventos donde el sistema invoca al manager y a los especialistas (con detalle solo en eventos seleccionados).
- Un resultado final consolidado con: **PLAN**, **APORTES POR ESPECIALISTA** y **RESPUESTA FINAL**.
- Identificación clara de qué texto viene de cada rol.

---

## 1️⃣ Configuración del Entorno de Trabajo

En esta sección configuraremos el entorno y verificaremos que todo está listo para el workshop.

In [1]:
import os
from dotenv import load_dotenv
from agent_framework import ChatAgent, MagenticBuilder, WorkflowOutputEvent
from agent_framework.openai import OpenAIChatClient

# Cargar variables de entorno
load_dotenv()

# Verificar configuración
print("=" * 72)
print("VERIFICACIÓN DEL ENTORNO DE TRABAJO")
print("=" * 72)

base_url = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model_id = os.getenv("AZURE_OPENAI_DEPLOYMENT")

print(f"✅ URL base Azure OpenAI: {'Configurada' if base_url else '❌ NO configurada'}")
print(f"✅ API Key: {'Configurada' if api_key else '❌ NO configurada'}")
print(f"✅ ID del modelo: {model_id if model_id else '❌ NO configurado'}")

if all([base_url, api_key, model_id]):
    print("\n✅ Entorno correctamente configurado - Listo para el workshop")
else:
    print("\n❌ Faltan variables de entorno - Revisa tu archivo .env")

VERIFICACIÓN DEL ENTORNO DE TRABAJO
✅ URL base Azure OpenAI: Configurada
✅ API Key: Configurada
✅ ID del modelo: gpt-5.4

✅ Entorno correctamente configurado - Listo para el workshop


---

## 2️⃣ Conceptos Fundamentales del Workshop

### 🎯 ¿Qué es el Patrón Magentic?

El patrón **Magentic** es una estrategia de orquestación de agentes basada en la investigación de **Magentic-One** de Microsoft. A diferencia de los patrones anteriores:

| Patrón | Flujo | Uso Ideal |
|--------|-------|----------|
| **Sequential** | A → B → C (Pipeline fijo) | Tareas paso a paso predefinidas |
| **Concurrent** | A \|\| B \|\| C (Todos simultáneos) | Perspectivas paralelas del mismo problema |
| **Magentic** | Gestor decide A/B/C dinámicamente | Tareas complejas con iteración adaptativa |

### 🔄 Componentes Clave del Patrón Magentic

```
┌─────────────────────────────────────┐
│   GESTOR MAGENTIC (Coordinación)    │
│   - Analiza el caso                 │
│   - Crea un plan de acción          │
│   - Selecciona qué especialista actúa│
│   - Evalúa progreso y replantea     │
└────────────┬────────────────────────┘
             │
        Itera en rondas:
             │
    ┌────────┴──────────┐
    │                   │
    ▼                   ▼
┌─────────────┐  ┌────────────┐
│ DIAGNÓSTICO │  │TRATAMIENTO │
└─────────────┘  └────────────┘
```

### 📊 Ciclo de Vida de una Orquestación Magentic

1. **Planificación**: El gestor analiza la tarea y crea un plan
2. **Selección**: El gestor elige qué agente debe actuar
3. **Ejecución**: El agente seleccionado ejecuta su trabajo
4. **Evaluación**: El gestor revisa el progreso
5. **Decisión**: Continuar, replantear o finalizar
6. **Iteración**: Volver a los pasos 2-5 hasta completar

---

## 3️⃣ Ejercicio Práctico

### 🎬 Crear tu Primer Gestor Magentic

En este ejercicio crearemos un equipo simple con un gestor Magentic que coordinará dos agentes.

**Paso 1**: Definir el gestor coordinador

In [2]:
# Paso 1: Crear el Gestor Coordinador
print("\n" + "=" * 72)
print("EJERCICIO 1: Crear tu Primer Gestor Magentic")
print("=" * 72)

gestor = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=base_url,
        api_key=api_key,
        model_id=model_id
    ),
    name="CoordinadorClinico",
    instructions="""Eres el Coordinador Clínico del hospital actuando como gestor Magentic:
1. Analiza el caso de forma clara (máximo 2-3 pasos)
2. Decide qué especialista debe actuar en cada etapa
3. Debes mostrar exactamente qué aportó cada especialista
4. Coordina hacia una respuesta segura y accionable

Formato obligatorio de salida final:
- PLAN:
  - Paso 1: ...
  - Paso 2: ...
- APORTES POR ESPECIALISTA:
  - [Diagnostico]: "<resumen literal de su aporte>"
  - [Tratamiento]: "<resumen literal de su aporte>"
- RESPUESTA FINAL:
  - <respuesta consolidada>

No omitas la sección "APORTES POR ESPECIALISTA".
Responde siempre en español."""
 )

print(f"✅ Gestor creado: {gestor.name}")
print("   Rol: Coordinador dinámico del equipo")


EJERCICIO 1: Crear tu Primer Gestor Magentic
✅ Gestor creado: CoordinadorClinico
   Rol: Coordinador dinámico del equipo


**Paso 2**: Crear agentes especialistas

In [3]:
# Paso 2: Crear Agentes Especialistas
especialista_1 = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=base_url,
        api_key=api_key,
        model_id=model_id
    ),
    name="Diagnostico",
    instructions="""Eres un especialista de Diagnóstico.
Tu rol:
- Identificar hipótesis y diagnósticos diferenciales
- Proponer 2-3 pruebas/acciones diagnósticas iniciales
- Señalar señales de alarma y riesgos

Responde siempre en español. Sé conciso y clínicamente prudente."""
 )

especialista_2 = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=base_url,
        api_key=api_key,
        model_id=model_id
    ),
    name="Tratamiento",
    instructions="""Eres un especialista de Tratamiento.
Tu rol:
- Proponer plan terapéutico inicial y medidas de soporte
- Considerar seguridad del paciente (contraindicaciones, vigilancia)
- Dar pasos concretos y priorizados

Responde siempre en español. Sé claro y accionable."""
 )

print(f"✅ Especialista 1: {especialista_1.name:15s} - Diagnóstico")
print(f"✅ Especialista 2: {especialista_2.name:15s} - Tratamiento")

✅ Especialista 1: Diagnostico     - Diagnóstico
✅ Especialista 2: Tratamiento     - Tratamiento


**Paso 3**: Construir el workflow Magentic

In [4]:
# Paso 3: Construir el Workflow Magentic
print("\n" + "-" * 72)
print("Construcción del Workflow Magentic")
print("-" * 72)

workflow_simple = (
    MagenticBuilder()
    .participants(
        analista=especialista_1,
        implementador=especialista_2
    )
    .with_standard_manager(
        agent=gestor,
        max_round_count=3,        # Máximo 3 iteraciones
        max_stall_count=1,        # Replanear después de 1 ronda sin progreso
        max_reset_count=1         # Veces que puede replantear desde cero (0 = no replantea)
    )
    .build()
)

print("✅ Workflow Magentic construido correctamente")
print("   - Gestor: Coordinador dinámico")
print("   - Agentes: Analista + Implementador")
print("   - Config: max_rounds=3, max_stalls=1, max_resets=1")


------------------------------------------------------------------------
Construcción del Workflow Magentic
------------------------------------------------------------------------
✅ Workflow Magentic construido correctamente
   - Gestor: Coordinador dinámico
   - Agentes: Analista + Implementador
   - Config: max_rounds=3, max_stalls=1, max_resets=1


**Paso 4**: Ejecutar el workflow y monitorear eventos

> 📌 **Nota**: Ejecuta la siguiente celda para ver el Magentic en acción. Observa cómo el gestor elige a qué agente invocar en cada ronda.

In [5]:
# Helpers reutilizables para trazas Magentic
EVENT_DESCRIPTIONS = {
    "WorkflowStartedEvent": "🚀 Workflow iniciado - Gestor comienza a planificar",
    "WorkflowStatusEvent": "📊 Estado - Evaluando progreso",
    "ExecutorInvokedEvent": "⚙️  Executor - Invocando acción",
    "WorkflowOutputEvent": "✅ Completado - Resultados agregados",
}


def _magentic_text_from_obj(obj):
    if obj is None:
        return None

    text = getattr(obj, "text", None)
    if isinstance(text, str) and text.strip():
        return text

    content = getattr(obj, "content", None)
    if isinstance(content, str) and content.strip():
        return content

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif hasattr(item, "text") and getattr(item, "text"):
                parts.append(getattr(item, "text"))
            else:
                parts.append(str(item))
        merged = " ".join(parts).strip()
        return merged or None

    delta = getattr(obj, "delta", None)
    if isinstance(delta, str) and delta.strip():
        return delta

    return None


def _magentic_author_and_text(event):
    candidates = [
        event,
        getattr(event, "data", None),
        getattr(event, "message", None),
        getattr(event, "update", None),
    ]

    for obj in candidates:
        if obj is None:
            continue

        author = (
            getattr(obj, "agent_name", None)
            or getattr(obj, "author_name", None)
            or getattr(obj, "name", None)
        )
        text = _magentic_text_from_obj(obj)
        if author or text:
            return author, text

    return None, None


def _magentic_executor_name(event):
    fields = [
        "executor_name",
        "participant_name",
        "target_name",
        "executor",
        "participant",
        "name",
    ]

    for field_name in fields:
        value = getattr(event, field_name, None)
        if isinstance(value, str) and value.strip():
            return value
        if value is not None and hasattr(value, "name"):
            nested = getattr(value, "name")
            if isinstance(nested, str) and nested.strip():
                return nested

    data = getattr(event, "data", None)
    if isinstance(data, dict):
        for field_name in fields:
            value = data.get(field_name)
            if isinstance(value, str) and value.strip():
                return value

    return None


def _extract_between(text, start_marker, end_markers):
    start = text.find(start_marker)
    if start < 0:
        return None

    start += len(start_marker)
    end = len(text)
    for marker in end_markers:
        idx = text.find(marker, start)
        if idx >= 0:
            end = min(end, idx)

    result = text[start:end].strip()
    return result or None


def _parse_aportes(aportes_text):
    lineas = [ln.strip() for ln in aportes_text.split("\n") if ln.strip()]
    aportes = []
    for ln in lineas:
        if ln.startswith("-") and "[" in ln and "]" in ln and ":" in ln:
            ini = ln.find("[")
            fin = ln.find("]", ini + 1)
            if ini >= 0 and fin > ini:
                nombre = ln[ini + 1:fin].strip()
                contenido = ln[fin + 1:].lstrip(": ").strip().strip('"')
                aportes.append((nombre, contenido))
    return aportes


def _magentic_log_system_event(event_num, event_type, event):
    if event_type == "ExecutorInvokedEvent":
        description = EVENT_DESCRIPTIONS.get(event_type, f"⚡ {event_type}")
        executor = _magentic_executor_name(event)
        if executor:
            print(f"  [{event_num:2d}] {'SISTEMA':15s} | {description} -> {executor}")
        else:
            print(f"  [{event_num:2d}] {'SISTEMA':15s} | {description}")
        return

    if event_type in {"WorkflowStartedEvent", "WorkflowStatusEvent", "WorkflowOutputEvent"}:
        description = EVENT_DESCRIPTIONS.get(event_type, f"⚡ {event_type}")
        agent_name = getattr(event, "agent_name", None) or getattr(event, "author_name", None)
        if agent_name:
            print(f"  [{event_num:2d}] {agent_name:15s} | {description}")
        else:
            print(f"  [{event_num:2d}] {'SISTEMA':15s} | {description}")


def _magentic_log_agent_update(
    event,
    last_text_by_agent,
    updates_by_agent,
    max_updates_per_agent=5,
    event_num=None,
    specialists_seen=None,
    show_raw_generation=False,
    raw_text_by_agent=None,
):
    author, text = _magentic_author_and_text(event)
    if not text:
        return

    author = author or "AGENTE"
    if specialists_seen is None:
        specialists_seen = set()

    # Modo RAW unificado: acumula por agente y no imprime token a token.
    if show_raw_generation:
        if raw_text_by_agent is not None:
            raw_text_by_agent[author] = text
        last_text_by_agent[author] = text
        updates_by_agent[author] = updates_by_agent.get(author, 0) + 1
        return

    previous = last_text_by_agent.get(author, "")
    printed = updates_by_agent.get(author, 0)

    if text.startswith(previous):
        new_chunk = text[len(previous):]
        show = len(new_chunk) >= 350
    else:
        new_chunk = text
        show = len(new_chunk.strip()) >= 120

    if not (show and printed < max_updates_per_agent):
        return

    # Manager: mostrar solo la parte de planificación para evitar ruido.
    if str(author).lower() in {"magentic_manager", "manager"}:
        plan_text = _extract_between(
            text,
            "- PLAN:",
            ["- APORTES POR ESPECIALISTA:", "- RESPUESTA FINAL:"],
        )
        if plan_text:
            print("      🧭 PLAN DEL MANAGER:")
            for ln in [x.strip() for x in plan_text.split("\n") if x.strip()]:
                print(f"         {ln}")
        else:
            print(f"      💬 {author}: {text.strip().replace(chr(10), ' ')}")

        last_text_by_agent[author] = text
        updates_by_agent[author] = printed + 1
        return

    # Especialistas: mostrar siempre la PRIMERA salida completa con su numero de evento.
    if author not in specialists_seen:
        full_text = text.strip().replace("\n", " ")
        if event_num is not None:
            print(f"      [{event_num:2d}] 🧩 {author}: {full_text}")
        else:
            print(f"      🧩 {author}: {full_text}")
        specialists_seen.add(author)
        last_text_by_agent[author] = text
        updates_by_agent[author] = printed + 1
        return

    # Especialistas (actualizaciones posteriores)
    if printed == 0:
        preview = text.strip().replace("\n", " ")
    else:
        preview = new_chunk.strip().replace("\n", " ")

    print(f"      💬 {author}: {preview}")
    last_text_by_agent[author] = text
    updates_by_agent[author] = printed + 1


def _magentic_log_executor_completed(event):
    """Imprime mensajes completos emitidos por especialistas al finalizar un executor."""
    data = getattr(event, "data", None)
    if not isinstance(data, list):
        return

    for msg in data:
        author = getattr(msg, "author_name", None) or getattr(msg, "agent_name", None)
        text = _magentic_text_from_obj(msg)
        if not author or not text:
            continue

        if str(author).lower() in {"magentic_manager", "manager"}:
            continue

        full_text = text.strip().replace("\n", " ")
        print(f"      🧩 {author}: {full_text}")


def _magentic_print_raw_unified(raw_text_by_agent):
    if not raw_text_by_agent:
        return

    print("\n🧪 GENERACIÓN RAW UNIFICADA (por agente):")
    for author, text in raw_text_by_agent.items():
        content = (text or "").strip().replace("\n", " ")
        if content:
            print(f"   - {author}: {content}")


def _magentic_print_results(output_evt):
    print("-" * 72)
    if not output_evt:
        return

    messages = output_evt.data
    if not messages:
        return

    # Normalmente el manager devuelve mensaje final consolidado en la primera posición.
    first = messages[0]
    author = first.author_name or "Asistente"
    content = first.text if hasattr(first, "text") else str(first.content)

    plan = _extract_between(content, "- PLAN:", ["- APORTES POR ESPECIALISTA:", "- RESPUESTA FINAL:"])
    aportes_text = _extract_between(content, "- APORTES POR ESPECIALISTA:", ["- RESPUESTA FINAL:"])
    respuesta = _extract_between(content, "- RESPUESTA FINAL:", [])

    print("\n✅ SALIDA ESTRUCTURADA:\n")

    print("1) MANAGER - PLAN")
    if plan:
        for ln in [x.strip() for x in plan.split("\n") if x.strip()]:
            print(f"   {ln}")
    else:
        print("   (No se pudo extraer el plan)")

    print("\n2) ESPECIALISTAS")
    aportes = _parse_aportes(aportes_text) if aportes_text else []
    if aportes:
        for nombre, aporte in aportes:
            print(f"   - {nombre}: {aporte}")
    else:
        print("   (No se pudieron extraer aportes individuales)")

    print("\n3) RESPUESTA GLOBAL")
    if respuesta:
        for ln in [x.strip() for x in respuesta.split("\n") if x.strip()]:
            print(f"   {ln}")
    else:
        print(f"   {content}")

    print(f"\nFuente final del manager: {author}")

In [6]:
# (Celda técnica) Referencias para evitar avisos del analizador estático en notebooks
_unused_static_refs = [
    _magentic_log_system_event,
    _magentic_log_agent_update,
    _magentic_log_executor_completed,
    _magentic_print_raw_unified,
    _magentic_print_results,
 ]

# No afecta a la salida del workshop
del _unused_static_refs

In [7]:
# Ejecutar el Workflow (modo clasico de trazas + detalle en eventos 1, 4 y 5)
async def ejecutar_ejercicio_1():
    tarea = (
        "Paciente de 54 años con fiebre alta, confusión y TA 85/55. "
        "Sospecha de sepsis. ¿Qué plan inicial (diagnóstico y tratamiento) propones?"
    )

    print(f"\n📋 TAREA: {tarea}\n")
    print("⏳ Ejecutando workflow Magentic...")
    print("-" * 72)

    event_descriptions = {
        "WorkflowStartedEvent": "🚀 Workflow iniciado - Gestor comienza a planificar",
        "WorkflowStatusEvent": "📊 Estado - Evaluando progreso",
        "ExecutorInvokedEvent": "⚙️  Executor - Invocando acción",
        "AgentRunUpdateEvent": "🤖 Agente - Procesando respuesta",
        "WorkflowOutputEvent": "✅ Completado - Resultados agregados",
    }

    # En estos eventos mostramos además el texto generado por agentes (si existe).
    eventos_con_detalle = {1, 4, 5}

    def extraer_texto_evento(event):
        if "_magentic_author_and_text" in globals():
            author, text = _magentic_author_and_text(event)
            return author, text

        candidates = [event, getattr(event, "data", None), getattr(event, "message", None), getattr(event, "update", None)]
        for obj in candidates:
            if obj is None:
                continue

            author = getattr(obj, "agent_name", None) or getattr(obj, "author_name", None) or getattr(obj, "name", None)
            text = getattr(obj, "text", None) or getattr(obj, "content", None) or getattr(obj, "delta", None)
            if isinstance(text, list):
                text = " ".join(str(x) for x in text)
            if author or text:
                return author, str(text) if text is not None else None

        return None, None

    output_evt = None
    evento_num = 0

    async for event in workflow_simple.run_stream(tarea):
        evento_num += 1
        event_type = type(event).__name__
        description = event_descriptions.get(event_type, f"⚡ {event_type}")

        agent_name = (
            getattr(event, "agent_name", None)
            or getattr(event, "author_name", None)
            or getattr(getattr(event, "data", None), "agent_name", None)
            or getattr(getattr(event, "data", None), "author_name", None)
        )

        if agent_name:
            print(f"  [{evento_num:2d}] {str(agent_name):15s} | {description}")
        else:
            print(f"  [{evento_num:2d}] {'SISTEMA':15s} | {description}")

        if evento_num in eventos_con_detalle:
            autor_detalle, texto_detalle = extraer_texto_evento(event)
            if texto_detalle and str(texto_detalle).strip():
                resumen = str(texto_detalle).replace("\n", " ").strip()
                actor = autor_detalle or "AGENTE"
                print(f"       ↳ {actor}: {resumen}")
            else:
                print("       ↳ (sin contenido textual en este evento)")

        if isinstance(event, WorkflowOutputEvent):
            output_evt = event
            break

    print("-" * 72)
    if output_evt:
        messages = list(output_evt.data or [])
        print(f"\n✅ RESULTADOS ({len(messages)} mensajes):\n")
        for i, msg in enumerate(messages, 1):
            autor = getattr(msg, "author_name", None) or "Asistente"
            contenido = msg.text if hasattr(msg, "text") else str(msg.content)
            print(f"[{i}] {autor}:")
            print(f"    {contenido}\n")

# Ejecutar el ejercicio
await ejecutar_ejercicio_1()


📋 TAREA: Paciente de 54 años con fiebre alta, confusión y TA 85/55. Sospecha de sepsis. ¿Qué plan inicial (diagnóstico y tratamiento) propones?

⏳ Ejecutando workflow Magentic...
------------------------------------------------------------------------
  [ 1] SISTEMA         | 🚀 Workflow iniciado - Gestor comienza a planificar
       ↳ (sin contenido textual en este evento)
  [ 2] SISTEMA         | 📊 Estado - Evaluando progreso
  [ 3] SISTEMA         | ⚙️  Executor - Invocando acción
  [ 4] magentic_manager | 🤖 Agente - Procesando respuesta
       ↳ magentic_manager: Paciente de 54 años con fiebre alta, confusión y TA 85/55. Sospecha de sepsis. ¿Qué plan inicial (diagnóstico y tratamiento) propones?
  [ 5] magentic_manager | 🤖 Agente - Procesando respuesta
       ↳ magentic_manager: We are working to address the following user request:  Paciente de 54 años con fiebre alta, confusión y TA 85/55. Sospecha de sepsis. ¿Qué plan inicial (diagnóstico y tratamiento) propones?   To answer this

Magentic Orchestrator: Max round count reached


  [2342] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2343] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2344] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2345] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2346] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2347] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2348] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2349] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2350] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2351] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2352] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2353] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2354] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2355] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2356] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2357] Tratamiento     | 🤖 Agente - Procesando respuesta
  [2358] Tratamiento     | 🤖 Agente - Procesando respues